<div style="width:100%; box-sizing:border-box; border-radius:16px; overflow:hidden; background-color:#1a1713; background-image:linear-gradient(rgba(18,15,11,0.74), rgba(18,15,11,0.74)), url('https://upload.wikimedia.org/wikipedia/commons/thumb/0/0c/Haeckel_Spumellaria.jpg/1280px-Haeckel_Spumellaria.jpg'); background-size:cover; background-position:center 30%;">
<div style="width:100%; max-width:860px; box-sizing:border-box; margin:0 auto; text-align:center; padding:48px 32px 40px 32px;">
<p style="margin:0; font-family:'Courier New', Courier, monospace; font-size:13px; font-weight:bold; letter-spacing:4px; color:#F0D68A; text-shadow:0 2px 12px rgba(0,0,0,0.95);">BIOHUB | CELL TRACKING DURING DEVELOPMENT</p>
<p style="margin:14px 0 0 0; font-family:Arial, sans-serif; font-size:42px; font-weight:900; line-height:1.12; color:#ffffff; text-shadow:0 2px 12px rgba(0,0,0,0.95);">The training labels, read across all 199 films</p>
<p style="margin:14px 0 0 0; font-family:'Courier New', Courier, monospace; font-size:12px; font-weight:bold; letter-spacing:2px; color:#F0D68A; text-shadow:0 2px 12px rgba(0,0,0,0.95);">ERNST HAECKEL | KUNSTFORMEN DER NATUR, PLATE 91, 1904 | PUBLIC DOMAIN</p>
<div style="margin-top:34px;">
<p style="margin:0; font-family:'Courier New', Courier, monospace; font-size:13px; font-weight:bold; letter-spacing:4px; color:#F0D68A; text-shadow:0 2px 12px rgba(0,0,0,0.95);">WHAT IS IN THE LABELS</p>
<div style="margin-top:12px;">
<div style="display:inline-block; vertical-align:top; width:190px; max-width:calc(100% - 24px); box-sizing:border-box; margin:12px; text-align:center;">
<p style="margin:0 0 8px 0; font-family:'Courier New', Courier, monospace; font-size:11px; font-weight:bold; letter-spacing:2px; color:#F0D68A; text-shadow:0 2px 12px rgba(0,0,0,0.95);">FILMS</p>
<p style="margin:0; font-family:Arial, sans-serif; font-size:30px; font-weight:900; line-height:1; color:#ffffff; text-shadow:0 2px 12px rgba(0,0,0,0.95);">199</p>
</div>
<div style="display:inline-block; vertical-align:top; width:190px; max-width:calc(100% - 24px); box-sizing:border-box; margin:12px; text-align:center;">
<p style="margin:0 0 8px 0; font-family:'Courier New', Courier, monospace; font-size:11px; font-weight:bold; letter-spacing:2px; color:#F0D68A; text-shadow:0 2px 12px rgba(0,0,0,0.95);">LABELLED DIVISIONS</p>
<p style="margin:0; font-family:Arial, sans-serif; font-size:30px; font-weight:900; line-height:1; color:#ffffff; text-shadow:0 2px 12px rgba(0,0,0,0.95);">151</p>
</div>
<div style="display:inline-block; vertical-align:top; width:190px; max-width:calc(100% - 24px); box-sizing:border-box; margin:12px; text-align:center;">
<p style="margin:0 0 8px 0; font-family:'Courier New', Courier, monospace; font-size:11px; font-weight:bold; letter-spacing:2px; color:#F0D68A; text-shadow:0 2px 12px rgba(0,0,0,0.95);">EMBRYOS</p>
<p style="margin:0; font-family:Arial, sans-serif; font-size:30px; font-weight:900; line-height:1; color:#ffffff; text-shadow:0 2px 12px rgba(0,0,0,0.95);">2</p>
</div>
<div style="display:inline-block; vertical-align:top; width:190px; max-width:calc(100% - 24px); box-sizing:border-box; margin:12px; text-align:center;">
<p style="margin:0 0 8px 0; font-family:'Courier New', Courier, monospace; font-size:11px; font-weight:bold; letter-spacing:2px; color:#F0D68A; text-shadow:0 2px 12px rgba(0,0,0,0.95);">CELLS WITH A LABEL</p>
<p style="margin:0; font-family:Arial, sans-serif; font-size:30px; font-weight:900; line-height:1; color:#ffffff; text-shadow:0 2px 12px rgba(0,0,0,0.95);">3.6%</p>
</div>
</div>
</div>
</div>
</div>

I read every training `.geff` and tabulated what is actually in them: how many cells are labelled,
how many divisions, and how that is distributed. Two things came out that I had not expected, and
both change how a local validation split should be built.

This reads graph arrays only, no image pixels, so it runs in a couple of minutes on CPU.

The competition metric is

$$\mathrm{score}=\mathrm{adj\_edge\_jaccard}+0.1\,\mathrm{division\_jaccard},$$
$$\mathrm{adj\_edge\_jaccard}=\max\left(0,J\left(1-0.1\frac{N_{\mathrm{pred}}-N_{\mathrm{est}}}{N_{\mathrm{est}}}\right)\right),$$

where J is edge Jaccard and the adjustment uses the predicted and estimated node counts. The board
shows only the sum, so two teams on the same total can need to fix opposite halves. A local split
reads both halves for free, which is why it is worth building one carefully.

One check before anything below: the division counter here, which counts nodes with two or more
outgoing edges, reproduces the official scorer's division count exactly. On fifteen films it returns
45 and the scorer's ledger for the same films is 45. Maximum out degree in these graphs is 2.

The chapter layout is borrowed from [cdeotte's EDA notebook](https://www.kaggle.com/code/cdeotte/fable-5-1-eda-original-data-insights)
for the current Playground Series, which reads far better than the flat list of sections I started
with. The closing chapter, where you paste your own split and get told what it can and cannot
resolve, is borrowed from [georgymamarin's ledger notebook](https://www.kaggle.com/code/georgymamarin/s6e9-what-the-board-paid-for-eleven-submissions)
in the same competition. Thanks to both.


## Chapter 1: Reading every label, and what one division event is worth

I first count labelled divisions. A division is a node with exactly two outgoing edges. These are graph labels, not a claim that all biological divisions were annotated.

The reader below follows the [GEFF specification](https://liveimagetrackingtools.org/geff/latest/specification/): the first edge column holds source IDs. It validates directedness, endpoints and duplicate edges before counting, and reads graph arrays only, never image pixels.

It also reads the Zarr v3 arrays directly rather than through the `zarr` package, because the Kaggle image does not ship `zarr`, `numcodecs` or `tracksdata` and a code competition runs with internet off. Each array here is a zstd-compressed chunk described by its own `zarr.json`, and `zstandard` is available. I checked the result against `zarr` on 597 arrays across all 199 training films and found no difference. Reuse it if you want a `.geff` reader that works in the stock image.

In [ ]:
from pathlib import Path
from collections import Counter
from math import comb
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zstandard

TRAIN = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
_ZDEC = zstandard.ZstdDecompressor()

def read_zarr_array(path):
    """Read one zarr v3 array without the zarr package.

    The Kaggle image ships zstandard but not zarr, numcodecs or tracksdata, and a code competition
    runs with internet off, so a notebook that imports zarr dies on its first cell. Every array in a
    .geff here is a zstd-compressed chunk described by its own zarr.json, which is little enough
    format to read directly. Checked against zarr on 597 arrays across all 199 training films: no
    difference.
    """
    meta = json.loads((path / 'zarr.json').read_text())
    if meta.get('zarr_format') != 3:
        raise ValueError(f"{path}: expected zarr v3, found {meta.get('zarr_format')}")
    codecs = [c['name'] for c in meta.get('codecs', [])]
    if codecs != ['bytes', 'zstd']:
        raise ValueError(f'{path}: unexpected codec chain {codecs}')
    endian = meta['codecs'][0].get('configuration', {}).get('endian', 'little')
    dtype = np.dtype(meta['data_type']).newbyteorder('<' if endian == 'little' else '>')
    shape = tuple(meta['shape'])
    chunks = tuple(meta['chunk_grid']['configuration']['chunk_shape'])
    sep = meta['chunk_key_encoding'].get('configuration', {}).get('separator', '/')
    out = np.full(shape, meta.get('fill_value', 0), dtype=dtype)
    if 0 in shape:
        return out
    expected = int(np.prod(chunks)) * dtype.itemsize
    for idx in np.ndindex(*tuple(int(np.ceil(s / c)) for s, c in zip(shape, chunks))):
        blob = path / ('c' + ''.join(sep + str(i) for i in idx))
        if not blob.exists():
            continue                      # a chunk that was never written keeps the fill value
        block = np.frombuffer(_ZDEC.decompress(blob.read_bytes(), max_output_size=expected),
                              dtype=dtype).reshape(chunks)
        region = tuple(slice(i * c, min((i + 1) * c, s)) for i, c, s in zip(idx, chunks, shape))
        out[region] = block[tuple(slice(0, r.stop - r.start) for r in region)]
    return out

def read_film(path):
    """One record per film, read from the graph arrays only. No image pixels are touched.

    A store that will not open should cost one film, not the whole table: a partial mirror shows up
    as missing arrays, and a reader that raises turns that into no output at all.
    """
    for part in ('nodes/ids', 'edges/ids', 'nodes/props/t/values'):
        if not (path / part / 'zarr.json').exists():
            raise ValueError(f'{path.name}: incomplete store, missing {part}')
    metadata = json.loads((path / 'zarr.json').read_text()).get('attributes', {}).get('geff', {})
    if not metadata:
        raise ValueError(f'{path.name}: incomplete store, no geff metadata')
    if metadata.get('directed') is not True:
        raise ValueError(f'{path.name}: expected an explicitly directed GEFF graph')

    nodes = read_zarr_array(path / 'nodes/ids')
    edges = read_zarr_array(path / 'edges/ids')
    frames = read_zarr_array(path / 'nodes/props/t/values')
    if nodes.ndim != 1 or len(np.unique(nodes)) != len(nodes):
        raise ValueError(f'{path.name}: invalid node IDs')
    if edges.ndim != 2 or edges.shape[1] != 2:
        raise ValueError(f'{path.name}: expected source/target edge pairs')
    if not np.isin(edges, nodes).all():
        raise ValueError(f'{path.name}: edge endpoint absent from nodes')
    if len(np.unique(edges, axis=0)) != len(edges):
        raise ValueError(f'{path.name}: duplicate edges')
    if np.any(edges[:, 0] == edges[:, 1]):
        raise ValueError(f'{path.name}: self edge')
    _, out_degree = np.unique(edges[:, 0], return_counts=True)
    # the organisers' own estimate of how many cells really exist, which is the denominator that
    # turns a labelled-node count into a labelled FRACTION
    estimated = (metadata.get('extra') or {}).get('estimated_number_of_nodes')
    return {
        'film': path.stem,
        'embryo': path.stem.split('_')[0],
        'divisions': int(np.count_nonzero(out_degree >= 2)),
        'labelled_nodes': int(len(nodes)),
        'labelled_edges': int(len(edges)),
        'estimated_nodes': float(estimated) if estimated else np.nan,
        'frames': int(frames.max()) + 1 if len(frames) else 0,
    }

paths = sorted(TRAIN.glob('*.geff'))
if not paths:
    raise FileNotFoundError(f'No GEFF stores found under {TRAIN}')
rows, skipped = [], []
for path in paths:
    try:
        rows.append(read_film(path))
    except Exception as error:
        skipped.append((path.stem, str(error)[:60]))
if not rows:
    raise RuntimeError('No readable GEFF store found')
films = pd.DataFrame(rows).set_index('film')
films['labelled_fraction'] = films.labelled_nodes / films.estimated_nodes
counts = films[['divisions']]
print('Films:', len(films), '| embryos:', dict(films.embryo.value_counts()),
      '| total labelled divisions:', int(films.divisions.sum()))
if skipped:
    print('Skipped', len(skipped), 'unreadable stores:')
    for film, error in skipped[:5]:
        print(' ', film, error)
print(films.head(5).to_string())


## Chapter 2: Two embryos, and the test set is neither of them

Folder names are `{embryo_id}_{field_of_view}`, so the first segment says which embryo a film came
from. There are exactly two in train. The competition description states that train and test are
embryo-disjoint, so the hidden test films come from an embryo that appears nowhere in training.

That matters for how a local split is built. Holding out random films validates inside the same two
embryos the model has already seen. The leaderboard asks a different question: does this transfer to
an embryo you have never seen? Those are not the same question, and a split that ignores the embryo
column cannot tell them apart.

In [ ]:
summary = films.groupby('embryo').agg(
    films=('divisions', 'size'),
    divisions=('divisions', 'sum'),
    films_with_a_division=('divisions', lambda s: int((s > 0).sum())),
    labelled_nodes=('labelled_nodes', 'sum'),
    labelled_edges=('labelled_edges', 'sum'),
    estimated_nodes=('estimated_nodes', 'sum'),
    median_frames=('frames', 'median'),
)
summary['share_of_labelled_nodes'] = summary.labelled_nodes / summary.labelled_nodes.sum()
summary['median_labelled_fraction'] = films.groupby('embryo').labelled_fraction.median()
print(summary.to_string(float_format=lambda v: f'{v:,.4f}'))

## Chapter 3: One embryo is labelled twelve times denser than the other

Each `.geff` carries `estimated_number_of_nodes`, the organisers' estimate of how many cells really
exist in that film. Dividing the labelled node count by it gives the fraction of cells that carry a
label, and that fraction is not remotely uniform.

On my run the medians are about 0.8 per cent for one embryo and about 9.7 per cent for the other, a
factor of roughly twelve, and roughly 85 per cent of all labelled nodes in the training set come
from the denser embryo alone even though it is 128 of the 199 films.

Two consequences worth keeping in mind. Edge Jaccard computed on a film from one embryo and on a
film from the other are not directly comparable, because the denominators differ by that factor.
And a model trained on the pooled set sees a labelled signal dominated by the denser embryo, while
the hidden test embryo has a density nobody outside the organisers knows.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

for embryo, group in films.groupby('embryo'):
    left.scatter(group.labelled_fraction * 100, group.divisions, s=18, alpha=0.7, label=embryo)
left.set_xscale('log')
left.set_xlabel('per cent of estimated cells that carry a label')
left.set_ylabel('labelled divisions in the film')
left.set_title('label density against divisions, per film')
left.legend(title='embryo')
left.grid(alpha=0.3)

order = sorted(films.embryo.unique())
data = [films.loc[films.embryo == e, 'labelled_fraction'].dropna() * 100 for e in order]
# matplotlib renamed this argument, so set the ticks by hand and stay version agnostic
right.boxplot(data)
right.set_xticks(range(1, len(order) + 1))
right.set_xticklabels(order)
right.set_yscale('log')
right.set_ylabel('per cent of estimated cells labelled')
right.set_title('label density by embryo')
right.grid(alpha=0.3, axis='y')

fig.tight_layout()
plt.show()

for embryo, group in films.groupby('embryo'):
    frac = group.labelled_fraction.dropna() * 100
    print(f'{embryo}: median {frac.median():.2f} per cent, range {frac.min():.2f} to {frac.max():.2f}')

For a subset with D labelled divisions, division Jaccard is TP / (TP + FP + FN). Starting from a perfect prediction, missing a division changes it by exactly 1/D. Adding a false division changes it by 1/(D+1). Near a good prediction, 1/D is a useful event-sensitivity approximation. Its weighted contribution to total score is about 0.1/D.

I call this an event-resolution check. It is not a confidence interval or a universal statistical noise floor. Below this scale, I inspect the event ledger before treating a small improvement as evidence of generalization. Edge contributions can still move by smaller amounts. With no labelled divisions, recall is untested, although false divisions can still matter. Scorer conventions for empty sets must also be checked.

The D values in the next cell are illustrative. The film table above gives the real ones.

In [ ]:
example_D = np.array([5, 8, 20, 45], dtype=int)
resolution = pd.DataFrame({
    'D': example_D,
    'missed event: division delta magnitude': 1 / example_D,
    'missed event: total delta magnitude': 0.1 / example_D,
    'false event: total delta magnitude': 0.1 / (example_D + 1),
})
print(resolution.to_string(index=False))

def noise_floor(films):
    films = list(films)
    if len(films) != len(set(films)):
        raise ValueError('Select each film only once')
    unknown = set(films) - set(counts.index)
    if unknown:
        raise ValueError(f'Unknown films: {sorted(unknown)}')
    D = int(counts.loc[films, 'divisions'].sum())
    return {'films': len(films), 'D': D,
            'division_event_sensitivity': 1 / D if D else np.nan,
            'total_event_sensitivity': 0.1 / D if D else np.nan,
            'status': 'labelled divisions available' if D else 'division recall untested'}

def select_films(division_counts, target_D):
    if not isinstance(target_D, (int, np.integer)) or target_D < 0:
        raise ValueError('target_D must be a nonnegative integer')
    ordered = sorted(division_counts.items(), key=lambda item: (-item[1], item[0]))
    if any(value < 0 or int(value) != value for _, value in ordered):
        raise ValueError('Division counts must be nonnegative integers')
    if sum(value for _, value in ordered) < target_D:
        raise ValueError('Target exceeds all available labelled divisions')
    chosen, total = [], 0
    for film, value in ordered:
        if total >= target_D:
            break
        chosen.append(film)
        total += int(value)
    return chosen

for target in example_D:
    try:
        chosen = select_films(counts.divisions.to_dict(), int(target))
        print('Target:', target, '| Selection:', chosen, '|', noise_floor(chosen))
    except ValueError as error:
        print('Target:', target, '|', error)
print('All films:', noise_floor(counts.index))

Taking the largest counts first minimizes the number of films needed to reach the target: no other selection of the same size can carry a larger total. This is only a division-coverage objective. I still check acquisition conditions, developmental stages, leakage between related films, and edge coverage. I freeze the split before tuning against it.

## Chapter 4: Picking films by division count, not at random

I measure skew instead of assuming it. The next cell reports zero-division films, low-count films, the median, and the largest film's share. It then calculates the exact distribution for a uniformly random four-film subset, without replacement. Dynamic programming counts subsets, so no random seed or Monte Carlo approximation is needed.

A zero-division subset is blind to division recall. A subset below a chosen target has coarse division-event sensitivity. Those statements are different. The probability table tests both and avoids claiming that random selection is usually blind unless the actual data support it.

In [ ]:
values = counts.divisions.to_numpy(dtype=int)
print(pd.Series({
    'films': len(values),
    'zero-division films': int((values == 0).sum()),
    'films with at most five divisions': int((values <= 5).sum()),
    'median divisions per film': float(np.median(values)),
    'maximum divisions per film': int(values.max()),
    'largest film share of divisions': float(values.max() / values.sum()) if values.sum() else np.nan,
}).to_string())

split_size = 4
if len(values) >= split_size:
    ways = [Counter() for _ in range(split_size + 1)]
    ways[0][0] = 1
    for seen, value in enumerate(values):
        for k in range(min(split_size, seen + 1), 0, -1):
            for total, number in list(ways[k - 1].items()):
                ways[k][total + int(value)] += number
    denominator = comb(len(values), split_size)
    assert sum(ways[split_size].values()) == denominator
    rows = [{'condition': 'D = 0', 'probability': ways[split_size][0] / denominator}]
    for target in example_D:
        rows.append({'condition': f'D < {target}',
                     'probability': sum(n for d, n in ways[split_size].items() if d < target) / denominator})
    print('Exact probabilities for a uniform four-film split:')
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print('Too few films for a four-film subset.')

ordered = counts.sort_values('divisions', ascending=False, kind='stable')
x = np.arange(len(ordered))
y = ordered.divisions.to_numpy(dtype=float)
sensitivity = np.divide(0.1, y, out=np.full_like(y, np.nan), where=y > 0)
fig, ax = plt.subplots(figsize=(max(10, len(ordered) * 0.28), 5))
ax.bar(x, y, color='steelblue', label='Labelled divisions')
ax.set(ylabel='Labelled divisions', xlabel='Film, sorted by division count')
ax.set_xticks(x)
ax.set_xticklabels(ordered.index, rotation=90, fontsize='small')
right = ax.twinx()
right.plot(x, sensitivity, 'o-', color='darkorange', label='Weighted event sensitivity')
right.set_ylabel('Approximate total-score sensitivity: 0.1 / D')
ax.set_title('Division coverage and single-event sensitivity by film')
fig.text(0.01, 0.01, 'Zero-division films have no plotted sensitivity: division recall is untested.')
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

The picture below is the whole argument in one frame. Left: how many films carry how many labelled
divisions, which is where the skew lives. Right: what one division event is worth in total score as a
ruler grows, which is the curve you are buying when you add films.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

edges = np.arange(values.min(), values.max() + 2) - 0.5
left.hist(values, bins=edges, color='#4c72b0', edgecolor='white')
left.set_xlabel('labelled divisions in a film')
left.set_ylabel('number of films')
left.set_title(f'{int((values == 0).sum())} of {len(values)} films carry none')
left.set_xticks(range(int(values.min()), int(values.max()) + 1))

grid = np.arange(1, int(values.sum()) + 1)
right.plot(grid, 0.1 / grid, color='#c44e52')
right.set_yscale('log')
right.set_xlabel('labelled divisions in the ruler, D')
right.set_ylabel('total score moved by one division event')
right.set_title('what one event is worth')
for mark in example_D:
    if mark <= grid[-1]:
        right.plot([mark], [0.1 / mark], 'o', color='#c44e52')
        right.annotate(f'D={mark}\n{0.1 / mark:.4f}', (mark, 0.1 / mark),
                       textcoords='offset points', xytext=(8, 6), fontsize=9)
right.grid(alpha=0.3)

fig.tight_layout()
plt.show()

print(f'A ruler of all {len(values)} readable films carries D = {int(values.sum())}, '
      f'so one division event moves the total by {0.1 / max(values.sum(), 1):.4f}.')

## Chapter 5: Getting the films without hitting the rate limit

I counted the files rather than guessing: each image store in my copies holds 102 files, roughly one
per timepoint plus metadata. Ten films pulled file by file is therefore about a thousand API
requests, which was enough to get me rate limited. A store chunked more finely would be much worse.

I pack selected films inside a separate Kaggle export kernel, then download its output archives. Each
archive holds the image store and its matching graph, which exchanges many small downloads for one
archive per film. Check disk capacity before running the export kernel. Packing does not shrink the
data or exempt you from download limits, it only reduces the number of requests.

This notebook prints the export snippet without executing it, so reading the ruler does not copy the
image dataset. Paste the printed snippet into the export kernel after choosing films.

In [ ]:
export_code = """from pathlib import Path
import tarfile

train = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
out = Path('/kaggle/working/film_archives')
out.mkdir(exist_ok=True)
selected_films = []  # Fill with film names from the count table.
if not selected_films:
    raise ValueError('Choose films before exporting')
for film in selected_films:
    store = train / (film + '.zarr')
    graph = train / (film + '.geff')
    if not store.is_dir() or not graph.exists():
        raise FileNotFoundError(f'Missing image or graph for {film}')
    archive = out / (film + '.tar')
    if archive.exists():
        raise FileExistsError(archive)
    with tarfile.open(archive, 'w') as tar:
        tar.add(store, arcname=store.name)
        tar.add(graph, arcname=graph.name)
    print(archive.name, archive.stat().st_size, 'bytes')
"""
print(export_code)

## Chapter 6: Checking a local split against the board, one axis at a time

For every real submission, I log the pipeline change, offline delta, and leaderboard delta. I keep the baseline and film selection fixed for each comparison. I label the axis by the intended intervention and separately record which metric components actually changed. An intervention can affect both components.

I count sign agreement separately for edge changes and division changes. Pooling them into an aggregate accuracy would hide the asymmetry I want to detect. A displayed tie does not establish the underlying sign.

| axis | change | offline reading | leaderboard | same sign |
|---|---|---|---|---|
| edge | turning off a relink stage | +0.0187 total, 5 films | 0.943 to 0.939 | no |
| edge | edge-feature TTA | -0.0002 adjusted edge, 4 films | 0.943 to 0.946 | no |
| edge | a division-CNN relaxation | -0.0058 total, 4 films | 0.946 to 0.946 | no |
| division | per-node division cost | +0.0079 total, 4 films | 0.946 to 0.948 | yes |
| division | raising the division gate | +0.0118 total, 4 films | 0.948 to 0.950 | yes |
| division | swapping the division network | -0.0058 total, 15 films | 0.950 to 0.949 | yes |

The offline column is not one consistent quantity, and I have named what each row actually was.
Early on I was reading the adjusted edge term, later the total. Mixing them is a defect in my own
ledger rather than a presentation choice, and it is part of why I now record which component moved
alongside the delta. The sign is what the count below uses, and the sign survives the inconsistency.

That is 0 for 3 on the edge axis and 3 for 3 on the division axis, scored throughout with the
competition's own scorer. The film sets are not identical across rows, since the ruler grew from 4
films to 5 and then to 15 while the ledger was being kept, and the rows say which. Six submissions
is a small record and I am not offering it as an estimate of future reliability. I am offering it as
the reason to keep the ledger at all: without it I would have gone on reading my edge numbers as if
they meant something.

One honest limit on reading that table. D counts labelled divisions, so 0.1/D sizes a division event
and says nothing about edge resolution. My edge-axis failures are therefore not explained by the
division event size, and I cannot tell from this record whether they came from resolution or from
bias. Separating those needs paired per-film component deltas and an edge-specific uncertainty, which
I do not have yet.

I retain the event ledger, inspect disagreement cases, and test the ruler on additional films before trusting its edge ranking. I treat the division agreement as encouraging evidence, not a guarantee.

## Chapter 7: A worked example on both halves of the metric

Several public notebooks delete nodes to work with the node-count term in the adjusted edge
Jaccard. I measured that on my own prediction dump, as an illustration of reading both halves of
the metric rather than one number. This is about my graph, not about anyone's notebook.

Measured on my fifteen-film ruler, applying it to my own prediction dump:

| | baseline | after the change |
|---|---|---|
| edge Jaccard | 0.8918 | 0.8918 |
| adjusted edge Jaccard | 0.8919 | 0.8919 |
| division Jaccard | 0.2097 | 0.1688 |
| division true / false / missed | 13 / 17 / 32 | 13 / 32 / 32 |
| total | 0.9129 | 0.9087 |

Both edge terms are unchanged to four decimals. The division Jaccard drops 0.0409, which at the
metric's weight of 0.1 is 0.0041 of total score, and that accounts for the whole 0.0042 gap to
rounding. Fifteen new false divisions, no change in either the true or the missed count. The asymmetry behind that is in how the scorer matches:
an unmatched predicted edge is dropped from consideration, while an unmatched predicted division is
penalised. Deleting nodes is therefore close to free on the edge axis and is not free on the
division axis.

Whether that holds for your pipeline depends on your graph, not on mine. The point of the example is
the shape of the check: compare both edge components and the full division ledger before and after,
rather than one headline number.

## Chapter 8: Provenance before adding public data

Public 3D microscopy of the same organism is easy to find and tempting as extra training data. Before
using any of it, compare its voxel scale and acquisition provenance against this competition's. The
competition's scale is 1.625 by 0.40625 by 0.40625 micrometres, and it is worth knowing that at least
one public zebrafish embryo store carries exactly that scale.

A matching scale does not establish that a dataset overlaps the held-out films. It does mean the
question is open, and an open question about leakage is worth resolving before training on the data
rather than after.

## Chapter 9: Check your own split

Paste your validation film names below. This reports what the split can resolve on the division
axis and whether it is balanced across the two embryos, which is the part that is easy to get wrong
by accident: films are not distributed evenly, so a random draw tends to land almost entirely in one
embryo, and the hidden test set is neither.

In [ ]:
MY_SPLIT = [
    # paste your held-out film names here, for example:
    # '6bba_05db0fb1', '6bba_07e24132', '44b6_12dfb391',
]

def check_split(names):
    names = list(names)
    if not names:
        return 'Nothing to check: fill MY_SPLIT with film names from the table above.'
    unknown = sorted(set(names) - set(films.index))
    if unknown:
        return f'Unknown films: {unknown}'
    if len(names) != len(set(names)):
        return 'A film appears twice in the split.'
    sub = films.loc[names]
    D = int(sub.divisions.sum())
    by_embryo = sub.groupby('embryo').agg(films=('divisions', 'size'), divisions=('divisions', 'sum'))
    lines = [f'{len(names)} films, {D} labelled divisions']
    lines.append(f'one division event moves the total score by '
                 + (f'{0.1 / D:.4f}' if D else 'nothing measurable, D = 0'))
    lines.append('')
    lines.append(by_embryo.to_string())
    lines.append('')
    if D == 0:
        lines.append('WARNING: no labelled divisions, so division recall is untested here. False '
                     'divisions still cost you, but you cannot see a missed one.')
    elif D < 20:
        lines.append(f'NOTE: at D = {D} a single division event is worth {0.1 / D:.4f} of total '
                     'score. Treat any offline difference smaller than that as unreadable.')
    if len(by_embryo) < 2:
        lines.append('WARNING: the split sits inside a single embryo. Train and test are '
                     'embryo-disjoint, so this measures transfer within an embryo the model has '
                     'already seen, which is an easier question than the board asks.')
    else:
        share = by_embryo.films / by_embryo.films.sum()
        # 0.75 rather than 0.8, because four films of five is the split people actually build by
        # accident and a strict greater-than lets exactly that case through
        if share.max() >= 0.75:
            lines.append(f'NOTE: {share.idxmax()} carries {100 * share.max():.0f} per cent of the '
                         'films here, so this behaves like a single-embryo split. My own five-film '
                         'ruler sits at four of five and I did not notice for weeks.')
    return '\n'.join(lines)

print(check_split(MY_SPLIT))